# Génération des exemples pour les traductions 
- à partir d'un fichier Phrases, le script remplace les déterminants variables pour formes des variantes de chaque phrase
- Phrases-B2.csv est le fichier de départ
- Phrases-B2-bis.csv est une extension pour ajouter des nouvelles traductions

In [1]:
# -*- coding: utf8 -*-
from os.path import expanduser
import codecs,yaml,re,random
import itertools as it
import ParFuMor as PFM
import pandas as pd

In [2]:
home = expanduser("~")
annee="25"
repertoire=home+"/sDrive/Cours/Bordeaux/L1-LinguistiqueGenerale/00-ProjetKalaba/"
nomTraductions=repertoire+"Phrases-B2-bis.csv"
nomLexique=repertoire+"%s-K1/Stems.yaml"%annee

In [3]:
with codecs.open(nomTraductions,"r",encoding="utf8") as input:
    traductionsIn=input.readlines()
with open(nomLexique, 'rb') as input:
    lexique=yaml.safe_load(input)

In [4]:
def getBottomList(dictLexique):
    listResult=[]
    for cle in dictLexique:
        if isinstance(dictLexique[cle],list):
            listResult.append(dictLexique[cle])
        else:
            listResult.extend(getBottomList(dictLexique[cle]))
    return listResult

def findFormes(formeIN):
    forme=formeIN.strip()
    for verbe in verbes:
        if forme in verbe:
            vsg=verbe[1::2]
            vpl=verbe[2::2]
            if forme in vsg:
                nombre="sg"
            elif forme in vpl:
                nombre="pl"
            else:
                print "problème avec la personne du verbe"
    return (nombre,vsg,vpl)

In [5]:
verbes=getBottomList(lexique["VER"])
findFormes("tombait")

('sg',
 ['tombe', 'tombait', 'tomba'],
 ['tombent', 'tombaient', u'tomb\xe8rent'])

In [6]:
def replaceDET(syntagme,nSaut=None):
    mots=re.split("(MS|MV|FS|FV|DU|DELA|AUX|AU|PL|DES)",syntagme)
    result=[mot if mot not in "MS|MV|FS|FV|DU|DELA|AUX|AU|PL|DES".split("|") or i==nSaut else random.choice(dAlternatives[mot]) for i,mot in enumerate(mots)]
    # print result
    return "".join(result).replace("' ","'").replace("  "," ")

def replacePhrase(phrase):
    lPhrase=[]
    syntagmes=phrase.split("\t")
    for n,syntagme in enumerate(syntagmes):
        if n in [0,2]:
            lPhrase.append(replaceDET(syntagme,1))
        elif n in [3,4,5]:
            lPhrase.append(replaceDET(syntagme),2)
        else:
            lPhrase.append(replaceDET(syntagme))
    return "\t".join(lPhrase)

def makeAlternative(alternative,n=20):
    result=set()
    for i in range(n):
        result.add(replaceDET(alternative))
    # print len(result)
    return result

In [7]:
dAlternatives={"MS":["le","ce","un"],
               "MV":["l'","cet","un"],
               "FS":["la","cette","une"],
               "FV":["l'","cette","une"],
               "AU":["au",u"à ce",u"à un"],
               "DU":["du","de ce","d'un"],
               "DELA":["de la","de cette","d'une"],
               "PL":["les","ces","des",
                     "les deux","ces trois","cinq","les six"],
               "AUX":["aux",u"à ces",u"à des",
                      u"à ces deux",u"aux trois",u"à trois",u"à cinq",u"à sept"],
               "DES":["des","de ces","de deux",
                      "des deux","de trois","de ces quatre","de treize"]}
def getAlternatives(chaine):
    result=set()
    resultSG=set()
    resultPL=set()
    alternatives=chaine.split(",")
    for alternative in alternatives:
        if alternative.split()[0] in ["MS","FS","MV","FV",
                                      "le","la","l","ce","cet","cette","un","une",
                                      "Nicole","Nabil","Katisha"]:
            resultSG=resultSG.union(makeAlternative(alternative.strip()))
        elif alternative.split()[0] in ["PL",
                                        "les","ces","des",
                                        "deux","trois","quatre","cinq","six"]:
            resultPL=resultPL.union(makeAlternative(alternative.strip()))
        else:
            result=result.union(makeAlternative(alternative.strip()))
        #     storeResult.add(alternative)
    return result,resultSG,resultPL         

In [8]:
findFormes("dormaient ")
getAlternatives(u"AUX villageois DELA forêt")

({u"aux villageois d'une for\xeat",
  u"\xe0 ces deux villageois d'une for\xeat",
  u'\xe0 ces deux villageois de cette for\xeat',
  u"\xe0 ces villageois d'une for\xeat",
  u'\xe0 ces villageois de cette for\xeat',
  u'\xe0 ces villageois de la for\xeat',
  u"\xe0 cinq villageois d'une for\xeat",
  u'\xe0 cinq villageois de cette for\xeat',
  u"\xe0 des villageois d'une for\xeat",
  u'\xe0 des villageois de la for\xeat',
  u"\xe0 sept villageois d'une for\xeat",
  u'\xe0 sept villageois de cette for\xeat',
  u'\xe0 trois villageois de la for\xeat'},
 set(),
 set())

In [9]:
phrasesOut=set()
verbes=getBottomList(lexique["VER"])
for ligne in traductionsIn[:]:
    syntagmes=ligne.strip().split("\t")
    alternatives={'':[""]}
    gnSG={}
    gnPL={}
    vSG=[]
    vPL=[]
    divers={}
    # print " ".join(syntagmes)
    (nombreSujet,vSG,vPL)=findFormes(syntagmes[1])
    if len(vSG)>2:
        vSG=vSG[:2]
    if len(vPL)>2:
        vPL=vPL[:2]
    for nSyntagme,syntagme in enumerate(syntagmes):
        if syntagme:
            if nSyntagme==0:
                gnSG,gnPL=getAlternatives(syntagme)[1:]
            elif nSyntagme==2:
                codSG,codPL=getAlternatives(syntagme)[1:]
                alternatives[nSyntagme]=codSG|codPL
            elif not nSyntagme in [1]:
                print syntagme
                alternatives[nSyntagme]=getAlternatives(syntagme)[0]
                print alternatives[nSyntagme]
    altSyntagmes=[list(alternatives[k]) if k in alternatives else [""] for k in range(2,len(syntagmes))]
    if gnSG:
        sujetSG=[gnSG,vSG]+altSyntagmes
        for phrase in it.product(*sujetSG):
            phrase=list(phrase)
            if not phrase[-1].endswith("."): phrase[-1]+="."
            phrasesOut.add("\t".join(phrase))
    if gnPL:
        sujetPL=[gnPL,vPL]+altSyntagmes
        for phrase in it.product(*sujetPL):
            phrase=list(phrase)
            if not phrase[-1].endswith("."): phrase[-1]+="."
            phrasesOut.add("\t".join(phrase))

avec PL lOUPs furieux DELA forêt
set([u'avec ces lOUPs furieux de la for\xeat', u"avec ces trois lOUPs furieux d'une for\xeat", u'avec cinq lOUPs furieux de la for\xeat', u'avec les deux lOUPs furieux de cette for\xeat', u'avec les deux lOUPs furieux de la for\xeat', u"avec ces lOUPs furieux d'une for\xeat", u'avec les six lOUPs furieux de cette for\xeat', u'avec ces lOUPs furieux de cette for\xeat', u'avec les six lOUPs furieux de la for\xeat', u"avec des lOUPs furieux d'une for\xeat", u'avec cinq lOUPs furieux de cette for\xeat', u'avec les lOUPs furieux de la for\xeat', u'avec ces trois lOUPs furieux de la for\xeat', u"avec les lOUPs furieux d'une for\xeat", u"avec les six lOUPs furieux d'une for\xeat"])
devant MS village DES démons
set([u'devant le village de deux d\xe9mons', u'devant un village de treize d\xe9mons', u'devant ce village de treize d\xe9mons', u'devant un village de ces quatre d\xe9mons', u'devant ce village de trois d\xe9mons', u'devant un village de deux d\xe9mons'

In [10]:
print "\n".join(getAlternatives(u"AUX enfANTs DELA forêt")[0])

à ces enfANTs d'une forêt
à ces deux enfANTs d'une forêt
à ces deux enfANTs de la forêt
aux trois enfANTs d'une forêt
à cinq enfANTs de la forêt
aux enfANTs de la forêt
à ces deux enfANTs de cette forêt
à ces enfANTs de cette forêt
à trois enfANTs de la forêt
à sept enfANTs d'une forêt
à sept enfANTs de cette forêt
aux trois enfANTs de la forêt


In [11]:
with codecs.open(nomTraductions.replace(".csv","-extend.csv"),"w",encoding="utf8") as output:
    for phrase in sorted(phrasesOut):
        output.write(phrase+"\n")

In [12]:
raise SystemExit("Stop right there!")

SystemExit: Stop right there!

/opt/anaconda3/envs/python2/lib/python2.7/site-packages/IPython/core/interactiveshell.py:2886: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


# Tirage aléatoire de phrases

In [ ]:
import random
lDet=["MS","FS"]
lNom=getBottomList(lexique["NOM"])
lAdj=getBottomList(lexique["ADJ"])
lVer=getBottomList(lexique["VER"])
lPrep=getBottomList(lexique["PREP"])
del lPrep[-2:]
del lPrep[10]
lAdj=[x for i,x in enumerate(lAdj) if i not in [1,10,11,12,20,21,24,29]]
for x in enumerate(lAdj):
    print(x)


In [ ]:
gn1=[lDet,lAdj,lNom]
gn2=[lDet,lAdj,lNom,lPrep,lDet,lAdj,lNom]
gn3=[lDet,lNom,["de"],lDet,lAdj,lNom,lPrep,lDet,lAdj,lNom]

In [ ]:
gns=[gn1,gn2,gn3]
gns=[gn1]
for gn in gns:
    for n in range(25):
        result=["" for s in gn]
        for i,mot in enumerate(gn):
            choix=random.choice(mot)
            if isinstance(choix,list):
                choix=choix[0]
            # print i,choix
            result[i]=choix
        result.append("\t")
        result.append(random.choice(lVer)[1])
        result.append("\t")
        for i,mot in enumerate(gn):
            choix=random.choice(mot)
            if isinstance(choix,list):
                choix=choix[0]
            # print i,choix
            result.append(choix)
        result.append("\t")
        result.append("\t")
        result.append(random.choice(lPrep)[0])
        for i,mot in enumerate(gn):
            choix=random.choice(mot)
            if isinstance(choix,list):
                choix=choix[0]
            # print i,choix
            result.append(choix)
        
            
        print " ".join(result).replace("\t ","\t")